In [1]:
"""
GDS with parallel x-oriented rectangles + corner alignment marks (L-shaped).
Alignment mark is its own cell, then referenced (copied/moved) to 4 corners.

Requires: gdspy  (pip install gdspy)
"""

import gdspy

gdspy.current_library = gdspy.GdsLibrary()
gds_lib = gdspy.GdsLibrary(unit=1e-6, precision=1e-9)


def make_L_mark_cell(
    lib: gdspy.GdsLibrary,
    cell_name: str = "ALIGN_L",
    arm_len: float = 20.0,
    arm_w: float = 2.0,
    layer: int = 10,
    datatype: int = 0,
):
    """
    Create an L-shaped mark centered at the origin corner (0,0) with arms extending +x and +y.
    """
    c = lib.new_cell(cell_name)

    # Horizontal arm: from (0,0) to (+arm_len, +arm_w)
    c.add(gdspy.Rectangle((0.0, 0.0), (arm_len, arm_w), layer=layer, datatype=datatype))
    c.add(gdspy.Rectangle((0.0, 0.0), (-arm_len, -arm_w), layer=layer, datatype=datatype))
    c.add(gdspy.Rectangle((0.0, 0.0), (arm_w, arm_len), layer=layer, datatype=datatype))
    c.add(gdspy.Rectangle((0.0, 0.0), (-arm_w, -arm_len), layer=layer, datatype=datatype))
    return c


def make_parallel_x_rectangles_with_corner_marks(
    outfile: str = "parallel_x_rectangles_with_corner_marks.gds",
    top_name: str = "TOP",
    n: int = 20,
    bar_length: float = 1000.0,
    bar_thickness: float = 1.0,
    pitch_y: float = 3.0,
    x0: float = 0.0,
    y0: float = 0.0,
    bar_layer: int = 1,
    bar_datatype: int = 0,
    # Marks
    mark_layer: int = 10,
    mark_datatype: int = 0,
    mark_arm_len: float = 20.0,
    mark_arm_w: float = 2.0,
    mark_offset: float = 25.0,
):
    if n <= 0:
        raise ValueError("n must be >= 1")

    top = gds_lib.new_cell(top_name)

    # --- Bars (x-oriented, stacked in y) ---
    for i in range(n):
        y_bot = y0 + i * pitch_y
        y_top = y_bot + bar_thickness
        top.add(
            gdspy.Rectangle(
                (x0, y_bot),
                (x0 + bar_length, y_top),
                layer=bar_layer,
                datatype=bar_datatype,
            )
        )

    # --- Compute pattern bounding box (bars only) ---
    minx, miny = x0, y0
    maxx = x0 + bar_length
    maxy = y0 + (n - 1) * pitch_y + bar_thickness

    # --- Alignment mark cell (one definition) ---
    align = make_L_mark_cell(
        gds_lib,
        cell_name="ALIGN_L",
        arm_len=mark_arm_len,
        arm_w=mark_arm_w,
        layer=mark_layer,
        datatype=mark_datatype,
    )

    # --- Place 4 references with rotations to make marks face inward/outward as you want ---
    # Here: the "inner corner" of each L is placed at a point offset outward from each bbox corner.
    # Since the base mark points +x/+y, we rotate it per corner and translate.

    placements = [
        # Bottom-left: base orientation (+x,+y)
        ((minx - mark_offset, miny - mark_offset), 0),
        # Bottom-right: rotate 90° so arms point (-y,+x) then translate
        ((maxx + mark_offset, miny - mark_offset), 90),
        # Top-right: rotate 180° so arms point (-x,-y)
        ((maxx + mark_offset, maxy + mark_offset), 180),
        # Top-left: rotate 270° so arms point (+y,-x)
        ((minx - mark_offset, maxy + mark_offset), 270),
    ]

    for (tx, ty), rot_deg in placements:
        ref = gdspy.CellReference(align, (tx, ty), rotation=rot_deg)
        top.add(ref)

    gds_lib.write_gds(outfile)
    return outfile


if __name__ == "__main__":
    make_parallel_x_rectangles_with_corner_marks(
        outfile="GDS/parallel_x_rectangles_with_corner_marks.gds",
        n=20,
        bar_length=1000.0,
        bar_thickness=1.0,
        pitch_y=3.0,
        x0=0.0,
        y0=0.0,
        bar_layer=1,
        bar_datatype=0,
        mark_layer=10,
        mark_datatype=0,
        mark_arm_len=20.0,
        mark_arm_w=2.0,
        mark_offset=25.0,
    )
    print("Wrote parallel_x_rectangles_with_corner_marks.gds")

Wrote parallel_x_rectangles_with_corner_marks.gds
